In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model,load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")

print("Path to dataset files:", path)

100%|██████████| 565M/565M [00:08<00:00, 73.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/paramaggarwal/fashion-product-images-small/versions/1


In [ ]:
df = pd.read_csv(path + "/styles.csv", nrows=5000, on_bad_lines='skip')
df['image'] = df.apply(lambda row: str(row['id']) + ".jpg", axis=1)
df = df.sample(frac=1).reset_index(drop=True)
df.head(10)

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image
0,35378,Women,Footwear,Shoes,Heels,Green,Summer,2012,Casual,Senorita Women Green Flats,35378.jpg
1,57758,Women,Personal Care,Skin Care,Face Wash and Cleanser,White,Spring,2017,NaN,Lotus Herbals Jojobawash Face Wash,57758.jpg
2,5434,Men,Apparel,Topwear,Tshirts,Red,Summer,2011,Sports,Nike Men's Kill Man Red T-shirt,5434.jpg
3,30868,Women,Apparel,Topwear,Kurtas,Off White,Summer,2012,Ethnic,Fabindia Women Off White Kurta,30868.jpg
4,17248,Men,Apparel,Topwear,Tshirts,Navy Blue,Fall,2011,Casual,U.S. Polo Assn. Men Stripes Navy Blue Polo Ts...,17248.jpg
5,59296,Men,Apparel,Topwear,Shirts,White,Fall,2012,Casual,U.S. Polo Assn. Men White & Green Shirt,59296.jpg
6,21521,Women,Accessories,Bags,Handbags,Pink,Winter,2015,Casual,Kiara Women Rose Pink Handbag,21521.jpg
7,37877,Unisex,Accessories,Socks,Socks,Cream,Summer,2016,Casual,Happy Socks Unisex Cream Socks,37877.jpg
8,27223,Women,Apparel,Dress,Dresses,Blue,Summer,2012,Casual,Doodle Kids Girl Printed Blue Dress,27223.jpg
9,26951,Women,Apparel,Topwear,Tshirts,Teal,Summer,2012,Casual,Jealous 21 Women Printed Teal T-shirt,26951.jpg


In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
df['image_path'] = df['image'].apply(lambda x: f'{path}/images/{x}')
print(f"Original dataset shape: {df.shape}")

# Display basic info
print("\nDataset info:")
print(df.info())
print(f"\nMissing values:\n{df.isnull().sum()}")

# Handle missing values
print("\nHandling missing values...")
# Drop rows with missing critical information
df = df.dropna(subset=['image', 'productDisplayName', 'masterCategory'])

# Fill missing values for other columns
df['subCategory'] = df['subCategory'].fillna('Unknown')
df['articleType'] = df['articleType'].fillna('Unknown')
df['baseColour'] = df['baseColour'].fillna('Unknown')
df['season'] = df['season'].fillna('Unknown')
df['usage'] = df['usage'].fillna('Unknown')

print(f"Dataset shape after cleaning: {df.shape}")

# Create target labels (we'll use masterCategory for classification)
label_encoder = LabelEncoder()
df['category_encoded'] = label_encoder.fit_transform(df['masterCategory'])
num_classes = len(label_encoder.classes_)
print(f"Number of categories: {num_classes}")
print(f"Categories: {label_encoder.classes_}")


Original dataset shape: (5000, 12)

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  5000 non-null   int64 
 1   gender              5000 non-null   object
 2   masterCategory      5000 non-null   object
 3   subCategory         5000 non-null   object
 4   articleType         5000 non-null   object
 5   baseColour          5000 non-null   object
 6   season              4999 non-null   object
 7   year                5000 non-null   int64 
 8   usage               4953 non-null   object
 9   productDisplayName  4999 non-null   object
 10  image               5000 non-null   object
 11  image_path          5000 non-null   object
dtypes: int64(2), object(10)
memory usage: 468.9+ KB
None

Missing values:
id                     0
gender                 0
masterCategory         0
subCategory          

In [ ]:
print("\n=== STEP 2: DATA AUGMENTATION & SPLITTING ===")

# Split the data (80% train, 12% test, 8% validation)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['category_encoded'])
test_df, val_df = train_test_split(temp_df, test_size=0.4, random_state=42, stratify=temp_df['category_encoded'])

print(f"Training set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"Validation set: {len(val_df)} samples")

# Image preprocessing and augmentation
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Data generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Function to create data generator from dataframe
def create_generator(dataframe, datagen, batch_size=BATCH_SIZE, shuffle=True):
    # Create a copy of dataframe with full image paths
    df_copy = dataframe.copy()
    df_copy['image_full_path'] = df_copy['image'].apply(lambda x: f'{path}/images/{x}')

    return datagen.flow_from_dataframe(
        df_copy,
        x_col='image_full_path',
        y_col='masterCategory',
        target_size=IMG_SIZE,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=shuffle
    )

# Create generators
print("Creating data generators...")
train_generator = create_generator(train_df, train_datagen)
val_generator = create_generator(val_df, val_test_datagen, shuffle=False)
test_generator = create_generator(test_df, val_test_datagen, shuffle=False)


=== STEP 2: DATA AUGMENTATION & SPLITTING ===
Training set: 3999 samples
Test set: 600 samples
Validation set: 400 samples
Creating data generators...
Found 3999 validated image filenames belonging to 5 classes.
Found 400 validated image filenames belonging to 5 classes.
Found 600 validated image filenames belonging to 5 classes.


In [ ]:
print("\n=== STEP 3: BUILD THE MODEL ===")

def create_model(num_classes):

    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    base_model.trainable = False

    inputs = base_model.input
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.2)(x)

    features = Dense(128, activation='relu', name='features')(x)

    predictions = Dense(num_classes, activation='softmax', name='predictions')(features)

    model = Model(inputs, predictions)

    return model


model = create_model(num_classes)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model architecture:")
model.summary()


=== STEP 3: BUILD THE MODEL ===
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model architecture:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ features (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
print("\n=== STEP 4: TRAIN THE MODEL ===")

EPOCHS = 3

# Fit the model
print("Starting model training...")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    verbose=1
)

# Save the model
model.save('fashion_search_model.h5')
print("Model saved as 'fashion_search_model.h5'")


=== STEP 4: TRAIN THE MODEL ===
Starting model training...


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 244s 2s/step - accuracy: 0.8457 - loss: 0.4637 - val_accuracy: 0.9625 - val_loss: 0.1471
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 242s 2s/step - accuracy: 0.9485 - loss: 0.1503 - val_accuracy: 0.9550 - val_loss: 0.1530
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 236s 2s/step - accuracy: 0.9681 - loss: 0.1058 - val_accuracy: 0.9750 - val_loss: 0.1172
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 247s 2s/step - accuracy: 0.9643 - loss: 0.1046 - val_accuracy: 0.9500 - val_loss: 0.1787
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 235s 2s/step - accuracy: 0.9690 - loss: 0.0993 - val_accuracy: 0.9750 - val_loss: 0.1216
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 241s 2s/step - accuracy: 0.9805 - loss: 0.0661 - val_accuracy: 0.9650 - val_loss: 0.1496
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 251s 2s/step - accuracy: 0.9756 - loss: 0.0806 - val_accuracy: 0.9825 - val_loss: 0.1115
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 242s 2s/step - accuracy: 0.9788 - loss: 0.0600 - val_accu

Model saved as 'fashion_search_model.h5'


In [7]:
print("\n=== STEP 5: MODEL EVALUATION ===")

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Calculate MAE (Mean Absolute Error) - for regression-like evaluation
# We'll use the predictions vs actual categories
test_predictions = model.predict(test_generator, verbose=0)
test_pred_classes = np.argmax(test_predictions, axis=1)
test_true_classes = test_generator.classes
mae = mean_absolute_error(test_true_classes, test_pred_classes)
print(f"Mean Absolute Error: {mae:.4f}")

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


=== STEP 5: MODEL EVALUATION ===


NameError: name 'model' is not defined

In [5]:
print("\n=== STEP 6: IMAGE SEARCH FUNCTIONALITY ===")

class FashionImageSearch:
    def __init__(self, model_path='fashion_search_model.h5'):
        # Load the trained model
        # self.model = tf.keras.models.load_model(model_path)
        self.model = model  # Use the model we just created

        # Create feature extraction model
        self.feature_model = Model(
            inputs=self.model.input,
            outputs=self.model.get_layer('features').output
        )

        self.image_features = {}
        self.product_data = df

    def preprocess_image(self, image_path):
        """Preprocess image for model input"""
        img = load_img(image_path, target_size=IMG_SIZE)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = img_array / 255.0
        return img_array

    def extract_features(self, image_path):
        """Extract features from an image"""
        img_array = self.preprocess_image(image_path)
        features = self.feature_model.predict(img_array, verbose=0)
        return features.flatten()

    def build_feature_database(self):
        """Build feature database for all products using /images folder"""
        print("Building feature database from /images folder...")
        success_count = 0
        error_count = 0

        for idx, row in self.product_data.iterrows():
            image_path = f"{path}/images/{row['image']}"

            if os.path.exists(image_path):
                try:
                    features = self.extract_features(image_path)
                    self.image_features[row['id']] = {
                        'features': features,
                        'product_info': row.to_dict()
                    }
                    success_count += 1
                except Exception as e:
                    print(f"Error processing {image_path}: {e}")
                    error_count += 1
            else:
                print(f"Image not found: {image_path}")
                error_count += 1

        print(f"Feature database built successfully!")
        print(f"- Processed: {success_count} images")
        print(f"- Errors: {error_count} images")

        # Save the feature database
        with open('image_features.pkl', 'wb') as f:
            pickle.dump(self.image_features, f)
        print("Feature database saved as 'image_features.pkl'")

    def load_feature_database(self):
        """Load pre-built feature database"""
        try:
            with open('image_features.pkl', 'rb') as f:
                self.image_features = pickle.load(f)
            print(f"Loaded feature database with {len(self.image_features)} images")
            return True
        except FileNotFoundError:
            print("No pre-built feature database found. Please build it first.")
            return False
    def search_similar_products(self, query_image_path, top_k=5):
        """Search for similar products given a query image"""
        if not self.image_features:
            print("Feature database is empty. Loading from file...")
            if not self.load_feature_database():
                return []

        # Extract features from query image
        query_features = self.extract_features(query_image_path)

        # Calculate similarities
        similarities = []
        for product_id, data in self.image_features.items():
            similarity = cosine_similarity(
                query_features.reshape(1, -1),
                data['features'].reshape(1, -1)
            )[0][0]

            similarities.append({
                'product_id': product_id,
                'similarity': similarity,
                'product_info': data['product_info']
            })

        # Sort by similarity (descending)
        similarities.sort(key=lambda x: x['similarity'], reverse=True)

        return similarities[:top_k]

    def display_search_results(self, results):
        """Display search results"""
        print(f"\nTop {len(results)} similar products:")
        print("-" * 80)

        for i, result in enumerate(results, 1):
            product = result['product_info']
            print(f"{i}. {product['productDisplayName']}")
            print(f"   Category: {product['masterCategory']} - {product['subCategory']}")
            print(f"   Color: {product['baseColour']}, Season: {product['season']}")
            print(f"   Similarity: {result['similarity']:.4f}")
            print(f"   Image: {product['image']}")
            print()



=== STEP 6: IMAGE SEARCH FUNCTIONALITY ===


In [6]:
print("\nInitializing search engine...")
search_engine = FashionImageSearch()

# Try to load existing database
if search_engine.load_feature_database():
    print("✓ Ready to search! You can now use search_similar_products()")
else:
    print("→ Run search_engine.build_feature_database() first to create the database")

# Example search function for easy use
def search_fashion_items(query_image_path, top_k=5):
    """Easy-to-use search function"""
    results = search_engine.search_similar_products(query_image_path, top_k)
    search_engine.display_search_results(results)
    return results

print("\n🎯 Quick search function available: search_fashion_items('your_image.jpg')")

# Save the label encoder for later use
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("\nLabel encoder saved as 'label_encoder.pkl'")


Initializing search engine...


NameError: name 'model' is not defined

In [1]:
def build_feature_database():
    """Build the feature database from your images - RUN THIS FIRST!"""

    print("Building feature database...")

    # Load the trained model
    model = load_model('fashion_search_model.h5')
    feature_model = Model(inputs=model.input, outputs=model.get_layer('features').output)

    image_features = {}
    success_count = 0
    error_count = 0

    for idx, row in df.iterrows():
        image_path = f"{path}/images/{row['image']}"

        if os.path.exists(image_path):
            try:
                # Process image
                img = load_img(image_path, target_size=(224, 224))
                img_array = img_to_array(img) / 255.0
                features = feature_model.predict(np.expand_dims(img_array, axis=0), verbose=0).flatten()

                # Store features
                image_features[row['id']] = {
                    'features': features,
                    'product_info': row.to_dict()
                }
                success_count += 1

                if success_count % 100 == 0:
                    print(f"Processed {success_count} images...")

            except Exception as e:
                print(f"Error processing {image_path}: {e}")
                error_count += 1
        else:
            error_count += 1

    # Save the database
    with open('image_features.pkl', 'wb') as f:
        pickle.dump(image_features, f)

    print(f"✅ Database built successfully!")
    print(f"   - Processed: {success_count} images")
    print(f"   - Errors: {error_count} images")
    print(f"   - Saved as: image_features.pkl")

In [2]:
def search_similar_images(query_image_path, top_k=5):
    """Simple function: input image → plot similar images with labels"""

    # Check if database exists, if not build it
    if not os.path.exists('image_features.pkl'):
        print("Feature database not found. Building it now...")
        build_feature_database()

    # Load model and feature database
    model = load_model('fashion_search_model.h5')
    feature_model = Model(inputs=model.input, outputs=model.get_layer('features').output)

    with open('image_features.pkl', 'rb') as f:
        image_features = pickle.load(f)

    print(f"Loaded database with {len(image_features)} products")

    # Process query image
    img = load_img(query_image_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    query_features = feature_model.predict(np.expand_dims(img_array, axis=0), verbose=0).flatten()

    # Find similarities
    similarities = []
    for product_id, data in image_features.items():
        similarity = cosine_similarity(
            query_features.reshape(1, -1),
            data['features'].reshape(1, -1)
        )[0][0]
        similarities.append((similarity, data['product_info']))

    # Get top results
    similarities.sort(reverse=True)
    top_results = similarities[:top_k]

    # Plot results
    fig, axes = plt.subplots(1, top_k + 1, figsize=(16, 4))

    # Query image
    axes[0].imshow(load_img(query_image_path))
    axes[0].set_title("Query Image", fontsize=10)
    axes[0].axis('off')

    # Similar images
    for i, (similarity, product_info) in enumerate(top_results):
        img_path = f"{path}/images/{product_info['image']}"
        if os.path.exists(img_path):
            axes[i+1].imshow(load_img(img_path))
        else:
            axes[i+1].text(0.5, 0.5, 'Not Found', ha='center', va='center')

        # Label with product name and similarity
        product_name = product_info['productDisplayName']
        if len(product_name) > 25:
            product_name = product_name[:22] + "..."

        label = f"{product_name}\nSimilarity: {similarity:.3f}"
        axes[i+1].set_title(label, fontsize=8)
        axes[i+1].axis('off')

    plt.tight_layout()
    plt.show()

    # Print text results too
    print(f"\n🔍 Top {top_k} similar products:")
    for i, (similarity, product_info) in enumerate(top_results, 1):
        print(f"{i}. {product_info['productDisplayName']} (Similarity: {similarity:.3f})")

    return top_results

In [3]:
search_similar_images(f'{path}/images/13849.jpg')

NameError: name 'path' is not defined